# Indian agriculture 1966-2017: cleaning the district data

**Notebook 1.** The data is ICRISAT's district-level file: 311 districts, 20 states, 52 years, with area, production and yield for different crops.

In this notebook I only clean the data and get it into a shape that's easy to analyse. The actual analysis comes in the next notebooks.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Loading the data

In [ ]:
district_data = pd.read_csv("../data/raw/ICRISAT-District_Level_Data.csv")
district_data.shape

In [ ]:
district_data.head()

In [ ]:
district_data.info()

Each row is one district in one year. The first 5 columns are ids (district code, year, state code, state name, district name) and the other 75 are area, production and yield for each crop.

## Missing values

In [ ]:
# pandas says nothing is missing
district_data.isnull().sum().sum()

Pandas says nothing is missing, but I don't trust it. Let me check the lowest values, because area, production and yield can't be negative.

In [ ]:
# the first 5 columns are ids, the rest are numbers
number_columns = district_data.columns[5:]

# lowest value in a few columns
district_data[["RICE AREA (1000 ha)", "MINOR PULSES PRODUCTION (1000 tons)", "OILSEEDS YIELD (Kg per ha)"]].min()

The lowest value is -1 in all three. So -1 must be how the file marks "no data".

In [ ]:
# how many cells are -1 in total
(district_data[number_columns] == -1).sum().sum()

In [ ]:
# which columns have the most
minus_one_count = (district_data[number_columns] == -1).sum()
minus_one_count.sort_values(ascending=False).head(10)

Most of the missing data is in oilseeds and minor pulses, plus a few smaller crops. Rice and wheat are barely affected.

Now I replace the -1 with NaN so pandas treats them as missing.

In [ ]:
district_data[number_columns] = district_data[number_columns].replace(-1, np.nan)

In [ ]:
# checking: missing values should now match the -1 count, and no negatives should be left
print("missing values:", district_data.isnull().sum().sum())
print("negative values:", (district_data[number_columns] < 0).sum().sum())

## Zeros

A lot of cells are 0. I think these are districts that just don't grow that crop. Cotton is a good example because it's only grown in some places.

In [ ]:
# share of rows with no cotton grown
(district_data["COTTON AREA (1000 ha)"] == 0).mean()

In [ ]:
cotton_yield = district_data["COTTON YIELD (Kg per ha)"]

print("average with zeros:", cotton_yield.mean())
print("average without zeros:", cotton_yield[cotton_yield > 0].mean())

Almost half the rows have no cotton, and those zeros pull the average yield down to about half of what it really is. So a zero yield where the area is also zero just means "not grown here". I'll turn those into NaN after reshaping the data.

## Reshaping the data

Right now every crop has its own 3 columns, which makes it hard to compare crops. I want one row per district, year and crop, with the columns `area`, `production` and `crop_yield`.

I'm leaving out oilseeds, fruits and vegetables, and the kharif/rabi sorghum columns. Oilseeds is a total of other crops, and sorghum already has a combined column, so keeping them would count the same land twice. Fruits, vegetables, potatoes, onion and fodder only have area, so I'm leaving those for now too.

In [ ]:
crop_names = ["Rice", "Wheat", "Sorghum", "Pearl millet", "Maize", "Finger millet",
              "Barley", "Chickpea", "Pigeonpea", "Minor pulses", "Groundnut",
              "Sesamum", "Rapeseed and mustard", "Safflower", "Castor",
              "Linseed", "Sunflower", "Soyabean", "Sugarcane", "Cotton"]

tables = []

for crop in crop_names:
    name = crop.upper()

    one_crop = district_data[["Dist Code", "Year", "State Name", "Dist Name"]].copy()
    one_crop["crop"] = crop
    one_crop["area"] = district_data[name + " AREA (1000 ha)"]
    one_crop["production"] = district_data[name + " PRODUCTION (1000 tons)"]
    one_crop["crop_yield"] = district_data[name + " YIELD (Kg per ha)"]

    tables.append(one_crop)

crop_data = pd.concat(tables, ignore_index=True)
crop_data.shape

In [ ]:
# shorter column names
crop_data = crop_data.rename(columns={"Dist Code": "dist_code", "Year": "year",
                                      "State Name": "state", "Dist Name": "district"})

# Orissa is called Odisha now
crop_data["state"] = crop_data["state"].replace("Orissa", "Odisha")

crop_data.head()

Units: `area` is in 1000 hectares, `production` is in 1000 tons and `crop_yield` is in kg per hectare.

## Zeros again, now in the new table

In [ ]:
# where the area is 0 the crop isn't grown, so the yield means nothing
not_grown = crop_data["area"] == 0
print("rows where the crop isn't grown:", not_grown.sum())

crop_data.loc[not_grown, "crop_yield"] = np.nan

## Checking the yield column

Yield should be production divided by area (times 1000 to get kg per hectare). I'll check if the file's yield follows that, and fill in missing yields where I can work them out.

In [ ]:
calculated_yield = crop_data["production"] / crop_data["area"] * 1000

difference = (calculated_yield - crop_data["crop_yield"]).abs() / crop_data["crop_yield"]

print("rows with a yield:", crop_data["crop_yield"].notna().sum())
print("rows where the yield is more than 2% off:", (difference > 0.02).sum())

In [ ]:
# rows where yield is missing but area and production are known
can_fill = crop_data["crop_yield"].isna() & (crop_data["area"] > 0) & crop_data["production"].notna()
print("yields I can fill in:", can_fill.sum())

crop_data.loc[can_fill, "crop_yield"] = calculated_yield[can_fill]

The yield matches production divided by area in about 99% of rows, so it's reliable.

## Strange values

In [ ]:
# highest yield for each crop
crop_data.groupby("crop")["crop_yield"].max().sort_values(ascending=False).head(8)

Chickpea at 58,000 kg/ha can't be right, a normal chickpea yield is around 1,000. Let me look at the highest yields (leaving out sugarcane, which is measured differently) together with their area.

In [ ]:
crop_data[crop_data["crop"] != "Sugarcane"].sort_values("crop_yield", ascending=False).head(5)

All of these have a tiny area (less than 1, which means under 1,000 hectares), so a small mistake in production turns into a huge yield. I'm not deleting them now, but when I compare yields later I'll leave out tiny areas.

Sugarcane's numbers are in a different range from real cane yields, so I'll check the units in ICRISAT's notes before using it.

## Final check and saving

In [ ]:
crop_data.info()

In [ ]:
# one last look at what's still missing, by crop
crop_data.groupby("crop")[["area", "production", "crop_yield"]].apply(lambda d: d.isnull().sum())

In [ ]:
os.makedirs("../data/clean", exist_ok=True)
crop_data.to_csv("../data/clean/crop_data_clean.csv", index=False)
print("saved:", crop_data.shape)

## What I did

- Found that -1 means "no data" and replaced it with NaN (13,132 cells).
- Found that zero area means "crop not grown", so I set yield to NaN there.
- Reshaped 80 columns into one row per district, year and crop.
- Left out totals and area-only columns so nothing is counted twice.
- Checked that yield matches production / area.
- Found a few impossible yields on tiny areas. I kept them for now and will leave out tiny areas when comparing yields.

The cleaned data is saved as `crop_data_clean.csv` and will be used in the next notebooks.